# 100-stock joint portfolio attribution

## Research question

Do the initial-allocation and subsequent-trading contributions found in the smaller universes persist in a larger, liquidity-selected universe? The attribution definitions and training protocol are shared with the 3- and 30-stock experiments, while data and outputs remain separate.

In [ ]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
from IPython.display import Image, display

START = Path.cwd().resolve()
ROOT = next((p for p in (START, *START.parents) if (p / "pyproject.toml").is_file()), None)
assert ROOT is not None, "Could not locate the project root."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.liquid100_extension import CONFIG, OUTPUT, REPORT, SEEDS, LABELS, load_extension_config, report, run_seeds
from dtasrl.data.inputs import load_real_inputs
from dtasrl.storage import verify_inventory
print(f"Python: {sys.version.split()[0]}")
print("Backend: SBX/JAX PPO; execution: next open")

## Inputs and provenance

Prepare data in the **100-stock universe** section of Notebook 00 first. This experiment only reads processed data and its frozen training-period scaler. It never downloads raw data or refits the scaler.

The candidate set is a frozen current S&P 500 public list crosschecked against IVV equity holdings. IVV is used only to crosscheck membership, not as an ETF return series or a traded asset. Eligibility uses pretraining history and training coverage. The 100 unique issuers are ranked by median Close × Volume over the final 252 training sessions ending in 2018. No test-period performance is used for selection.

This is a custom liquid 100-stock universe, not the S&P 100. Current membership is retrospective and has survivorship bias. It does not represent historical index membership.

In [ ]:
cfg = load_extension_config()
frozen = ROOT / cfg["data"]["frozen_inputs"]
verify_inventory(frozen)
prepared, data_hash = load_real_inputs(cfg)
snapshot = json.loads((ROOT / cfg["data"]["snapshot"]).read_text(encoding="utf-8"))
selection = pd.read_parquet(ROOT / cfg["data"]["selection_audit"])
display(pd.DataFrame([
    {"item": "Candidate-list retrieval", "value": snapshot["membership_sources"]["constituents"]["retrieved_at_utc"]},
    {"item": "Holdings crosscheck date", "value": snapshot["candidate_crosscheck"]["holdings_as_of"]},
    {"item": "Price cutoff", "value": snapshot["date_max"]},
    {"item": "Processed SHA-256", "value": data_hash},
    {"item": "Selection SHA-256", "value": snapshot["selection_sha256"]},
    {"item": "Frozen inputs", "value": cfg["data"]["frozen_inputs"]},
]))
display(selection.loc[selection.selected].sort_values("rank")[["rank", "ticker", "company", "sector", "training_coverage", "median_training_dollar_volume"]].head(20))
print("The full 100-stock list and all candidate exclusion decisions are saved locally.")

## Data checks

The first row of each split is the previous-close observation. Only subsequent rows produce in-split returns. Features are ordered feature-major, then alphabetically by ticker. All eight original features and the original accounting convention are retained.

In [ ]:
assert len(cfg["tickers"]) == len(cfg["observation_tickers"]) == 100
assert cfg["tickers"] == sorted(cfg["tickers"])
assert selection.loc[selection.selected, "issuer_id"].nunique() == 100
assert snapshot["date_max"] == "2026-08-31"
rows = []
for name in ("train", "validation", "test"):
    market = prepared[name]
    assert market["tickers"] == cfg["tickers"]
    assert market["prices"].shape[1] == 100
    assert market["features"].shape[1] == 800
    assert np.isfinite(market["features"]).all()
    assert np.isfinite(market["prices"]).all() and (market["prices"] > 0).all()
    assert len(market["dates"]) > 1
    rows.append({"split": name, "first observation": str(market["dates"][0])[:10],
                 "last observation": str(market["dates"][-1])[:10],
                 "evaluation/training transitions per episode": len(market["dates"]) - 1,
                 "stocks": 100, "features": 800, "observation dimension": 901, "action dimension": 101})
display(pd.DataFrame(rows))

## Configuration

Training: 2009–2018. Validation: 2019. Main test: 2020–2025. The continuous 2026 path through August 31 is an extension, reported separately. Three seeds run sequentially, each for 5,001,216 transitions. The model architecture, PPO hyperparameters, reward scaling, transaction costs, and target-weight environment match the Dow 30 extension.

In [ ]:
from dtasrl.config import load_config
original = load_config(ROOT / "configs/experiments/multi_dow30.yaml")
for key in ("model", "split", "total_timesteps", "temperature", "execution", "reward_scale", "transaction_cost_bps"):
    assert cfg[key] == original[key], f"Unexpected protocol difference: {key}"
display(pd.DataFrame([{"parameter": key, "value": str(cfg[key])}
                      for key in ("algorithm", "backend", "model", "total_timesteps", "transaction_cost_bps", "execution", "output_group")]))
print("Displayed seed IDs:", LABELS)
print("No training has been started by the data preparation or report functions.")

## Execute experiments

**Manual training switch.** Running the following cell with `RUN_EXPERIMENTS = True` starts the three real PPO workers sequentially. Set it to `False` for report-only use. Completed, verified runs are skipped. Incomplete runs resume only with a complete checkpoint; unsafe or corrupt runs stop for inspection. There is no automatic deletion.

Keep at most two formal training workers active across all notebooks and retain at least 8 GiB of available memory. This extension must not be started alongside two existing workers. Interrupting this cell stops its own child worker, not other experiments. Logs, checkpoints, models and full evaluation records are written to `outputs/05_liquid100`.

In [ ]:
RUN_EXPERIMENTS = False
if RUN_EXPERIMENTS:
    execution_log = run_seeds()
    display(execution_log)
else:
    print("Report-only execution. No training process was launched.")

## Results

The controls are unchanged: full learned policy, initial-allocation hold, initial-weight rebalance, equal-weight hold, cash, exposure-matched equal-weight hold, Freeze-1/5/20/63, and the four exposure/composition paths. Fixed-decision cost sensitivity uses 0/5/10/25 bps and is not retraining.

Initial allocation A is log wealth of Freeze-1 minus equal-weight hold. Subsequent trading D is full-policy log wealth minus Freeze-1. A + D must equal the full-policy excess log wealth. Exposure/composition Shapley terms are retrospective path attribution.

Only complete, identity-matched runs enter the result tables. Missing results are shown as not_started or incomplete, without fabricated metrics.

In [ ]:
result = report()
display(result["status"][["condition", "seed_id", "status", "run"]])
if result["summary"].empty:
    print("No completed 100-stock models yet. Training is required before interpreting results.")
else:
    summary = result["summary"].drop(columns=["seed"], errors="ignore")
    display(summary[summary.strategy.isin(["full_ppo", "equal_weight_hold", "freeze_1", "freeze_5", "freeze_20", "freeze_63"])])
    display(result["effects"].drop(columns=["seed"], errors="ignore"))

## Figures

PNG figures use 320 DPI. Wealth curves have no point markers and use thin lines. Allocation displays aggregate the small initial positions and show the 20 largest mean holdings for readability; every stock's complete weights remain available in the saved CSV and Parquet files.

In [ ]:
for path in result["figures"]:
    display(Image(filename=str(path)))

## Interpretation and limitations

Compare policy and controls **within this 100-stock universe** first. Agreement or disagreement with the three-stock and Dow 30 findings provides descriptive evidence of robustness to a broader investment set. It is not an isolated causal effect of asset count: the companies, diversification opportunities and action/observation dimensions also change.

The three seeds measure learning randomness, not three independent market histories. The stock pool remains retrospective. Fixed-action cost replay and test-mean composition paths are attribution analyses rather than independently trained, deployable policies. Do not select a seed, Freeze-K or transaction cost because its result is more favorable.

## Output inventory

Data and selection: flat `data/raw/liquid100_*`, `data/interim/liquid100_*`, `data/processed/liquid100_*`, with frozen arrays in `data/processed/inputs/liquid100`.

Models and step records: `outputs/05_liquid100/runs/ppo_multi_liquid100/<config-hash>/<seed>/`.

Human-readable report: `outputs/05_liquid100/report/`. Each run retains the source snapshot, configuration, dependency versions, training statistics, checkpoints, complete validation/test observations and actions, target/pretrade/closing weights, trade values, rewards, costs, wealth, counterfactual paths and hash inventory. Operations in this notebook do not write into the previous experiment groups.

In [ ]:
display(pd.read_csv(REPORT / "inventory.csv")[["path", "bytes"]])
print("Reports and their plotting data persist locally. No result depends on a live notebook kernel.")